23070521135 - Sharvayu Zade 

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Load the dataset from the same folder as this notebook
data = pd.read_csv("creditcard.csv")

X = data.drop(columns="Class")
y = data["Class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using only the training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Give more importance to the rare fraud class
weights = compute_class_weight(
    class_weight="balanced", classes=[0, 1], y=y_train
)
class_weights = {0: weights[0], 1: weights[1]}

print(f"Dataset shape: {data.shape}")
print(f"Fraud transactions: {y.sum()} ({y.mean() * 100:.2f}%)")
print(f"Class weights: {class_weights}")

Dataset shape: (284807, 31)
Fraud transactions: 492 (0.17%)
Class weights: {0: 0.5008661206149896, 1: 289.14340101522845}


In [2]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Build a simple deep neural network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=2048,
    class_weight=class_weights,
    verbose=1
)

# Convert probabilities to class predictions
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Genuine", "Fraud"], zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7560 - loss: 0.6121 - val_accuracy: 0.8713 - val_loss: 0.4773
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9120 - loss: 0.3059 - val_accuracy: 0.9710 - val_loss: 0.2799
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9641 - loss: 0.2451 - val_accuracy: 0.9804 - val_loss: 0.1949
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9673 - loss: 0.2098 - val_accuracy: 0.9760 - val_loss: 0.1771
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9727 - loss: 0.1855 - val_accuracy: 0.9756 - val_loss: 0.1561
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9697 - loss: 0.1710 - val_accuracy: 0.9773 - val_loss: 0.1309
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9696 - loss: 0.1618 - val_accuracy: 0.9769 - val_loss: 0.1252
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9732 - loss: 0.1490 - val_accuracy: 0.9780 - val_loss: